# Grouping and Pivot Tables

This lab goes with two chapters: "Grouping" (parts 1 to 4) and "Pivot Tables and Crosstabs" (parts 5 and 6).

- Part 1 — Example data `sales`: the output shapes of `agg`, `transform`, `filter` and `apply`, ranks and running totals, checkable by hand
- Part 2 — Aggregations: the price per m² of homes, by département
- Part 3 — Hierarchical indexes: regions and départements
- Part 4 — `transform`, `filter` and `apply`: deviation from the median, outlier sales, most expensive sales
- Part 5 — Pivot tables: the price per m² by region and by year
- Part 6 — Crosstabs and `melt`: the payment defaults of a bank's clients

Parts 2 to 5 build on the same data: each one uses the variables of the previous ones, which each exercise's solution defines. The exercises marked "optional" aren't needed afterwards: skip them if time runs short.

## Data and Libraries

- [DVF](https://www.data.gouv.fr/fr/datasets/demandes-de-valeurs-foncieres-geolocalisees/) (*Demandes de valeurs foncières*, Etalab, Licence Ouverte 2.0): the real estate sales in France from January 2014 to June 2022, one row per transaction, in the cleaned version of [NyxAether/DVF](https://github.com/NyxAether/DVF), as Parquet partitioned by year.
- [Départements de France](https://www.data.gouv.fr/datasets/departements-de-france) (data.gouv.fr, ODbL): each French département and its region.
- [Default of Credit Card Clients](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients) (UCI, CC BY 4.0): 30,000 clients of a Taiwanese bank in 2005.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh dvf
!wget -q -nc https://github.com/shuuchuu/datasets/raw/refs/heads/main/france-departements/departements-france.csv
!wget -q -nc https://github.com/shuuchuu/datasets/raw/refs/heads/main/credit-default/credit-default.csv

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

## Part 1 — Example Data `sales`

The slides' dataset, one row per sale: small enough to check every result by hand. `shape_of` prints what to look at in a result: its type, its shape (rows, columns), the names of its index levels and its first three labels.

In [ ]:
sales = pd.DataFrame(
  [
    ("North", "Lille", "bike", "2026-01", 3, 1500),
    ("North", "Lille", "helmet", "2026-01", 10, 400),
    ("North", "Amiens", "bike", "2026-02", 2, 900),
    ("South", "Nice", "bike", "2026-01", 5, 2600),
    ("South", "Nice", "helmet", "2026-02", 8, 300),
    ("South", "Marseille", "helmet", "2026-02", 6, 250),
    ("West", "Nantes", "bike", "2026-01", 4, 2000),
    ("West", "Rennes", "helmet", "2026-02", 7, 290),
    ("West", "Nantes", "helmet", "2026-02", 3, 120),
    ("North", "Amiens", "helmet", "2026-01", 5, 200),
  ],
  columns=["region", "city", "product", "month", "quantity", "amount"],
)


def shape_of(result):
  """Type, shape, levels and start of the index of result."""
  index = result.index
  print(type(result).__name__, result.shape, list(index.names), list(index[:3]))


sales

### 1.1 Predict the Output Shapes

With `g = sales.groupby("region")`, six computations:

1. `g["amount"].sum()`
2. `g.agg(n=("amount", "size"), revenue=("amount", "sum"))`
3. `g["amount"].transform("sum")`
4. `g.head(1)`
5. `g.filter(lambda d: len(d) > 3)`
6. `g.apply(lambda d: d.nlargest(2, "amount"), include_groups=False)`

For each one, first predict, without running it: `Series` or `DataFrame`, the number of rows and columns, and the index (the regions, the original index of `sales`, or both). Write your answers down as comments, then pass each result to `shape_of`.

Done when your six predictions are right; if they aren't, read the "Output Shapes" frame again before going on.

In [ ]:
# Your code here

#### Additional hints

The rules from the slides:

- one row per group, indexed by the key: aggregations (`sum`, `agg`);
- the original index, every row: transformations (`transform`, `cumsum`, `rank`);
- the original index, some of the rows: filters (`head`, `filter`);
- `apply`: depends on what the function returns; a `DataFrame` per group gives their concatenation, with the key as an index level added before the original index;
- `include_groups=False`: the key column isn't passed to the function, so it isn't in the result.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
g = sales.groupby("region")
shape_of(g["amount"].sum())
shape_of(g.agg(n=("amount", "size"), revenue=("amount", "sum")))
shape_of(...)
```

#### Solution

In [ ]:
g = sales.groupby("region")
shape_of(g["amount"].sum())  # Series (3,): one value per region
shape_of(g.agg(n=("amount", "size"), revenue=("amount", "sum")))  # DataFrame (3, 2)
shape_of(g["amount"].transform("sum"))  # Series (10,): original index
shape_of(g.head(1))  # DataFrame (3, 6): rows 0, 3 and 6 of sales
shape_of(g.filter(lambda d: len(d) > 3))  # DataFrame (4, 6): North only
# DataFrame (6, 5): MultiIndex (region, original index), without the region column
shape_of(g.apply(lambda d: d.nlargest(2, "amount"), include_groups=False))

### 1.2 Named Aggregations by Product

For each product, compute in a single `agg`, with named aggregations, the number of sales `n`, the quantity sold `quantity` and the revenue `revenue`. Then add the `unit_price` column: the revenue divided by the quantity. Also compute, by product, the mean of each sale's unit price (its amount divided by its quantity): why does it differ from `unit_price`?

To check: 6 helmet sales, 39 helmets for 1,560 €, that is 40 € per helmet; 500 € per bike, but 492.50 € as the mean of its four sales' prices.

In [ ]:
# Your code here

#### Additional hints

The calls and parameters to use:

- `sales.groupby("product").agg(name=(column, function), ...)`: one column per named aggregation, in the order written;
- `size` counts the group's rows, `sum` adds up its values;
- `assign(unit_price=...)`: the ratio of two sums is computed after the aggregation;
- `sales.assign(sale_price=sales["amount"] / sales["quantity"])`: the unit price of each sale, before the aggregation, then `groupby("product")["sale_price"].mean()`; this mean gives the same weight to every sale, whatever its quantity, and `unit_price` to every item sold.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
by_product = sales.groupby("product").agg(
  n=("amount", ...),
  quantity=(..., "sum"),
  revenue=...,
)
print(by_product.assign(unit_price=...))
by_sale = sales.assign(sale_price=...)
by_sale.groupby(...)["sale_price"].mean()
```

#### Solution

In [ ]:
by_product = sales.groupby("product").agg(
  n=("amount", "size"),
  quantity=("quantity", "sum"),
  revenue=("amount", "sum"),
)
print(by_product.assign(unit_price=by_product["revenue"] / by_product["quantity"]))
# Every sale weighs as much as any other in the mean, whatever its quantity
by_sale = sales.assign(sale_price=sales["amount"] / sales["quantity"])
by_sale.groupby("product")["sale_price"].mean()

### 1.3 Ranks and Running Totals

Add to a copy of `sales` (`assign`) three columns, computed within each row's group:

- `region_rank`: the sale's rank in its region, 1 for the largest, with `rank`;
- `product_total`: the running revenue of its product, in month order, with `cumsum`;
- `unsorted_total`: the same running total, computed without sorting first.

Print the result sorted by product and by month: where do both running totals differ, and why?

To check: the Nice bike sale is the first in the South, the Amiens helmet (200 €) the fourth in the North; each product's last running total is its whole revenue (7,000 € for bikes, 1,560 € for helmets); the Amiens bikes, the only bike sale in February, total 7,000 € after the sort, 2,400 € without.

In [ ]:
# Your code here

#### Additional hints

The calls and parameters to use:

- `sales.groupby("region")["amount"].rank(ascending=False)`: 1 for the largest amount of each region; floats, since two ties would share an average rank (1.5);
- `sales.sort_values(["product", "month"])`, then `groupby("product")["amount"].cumsum()`: a running total follows the order of the rows, hence the sort first;
- the result keeps the original index: `assign` aligns it back with the rows of `sales`, whatever the order of the sort;
- without a sort, the running total follows the table's order, where the Amiens helmet, sold in January, comes after the February sales.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
ranks = sales.groupby(...)["amount"].rank(ascending=...)
sorted_sales = sales.sort_values([..., ...])
total = sorted_sales.groupby(...)["amount"].cumsum()
unsorted = sales.groupby(...)["amount"].cumsum()
with_totals = sales.assign(region_rank=ranks, product_total=..., unsorted_total=...)
with_totals.sort_values(["product", "month"])
```

#### Solution

In [ ]:
ranks = sales.groupby("region")["amount"].rank(ascending=False)
sorted_sales = sales.sort_values(["product", "month"])
total = sorted_sales.groupby("product")["amount"].cumsum()
# Without a sort: the table's order, where a January sale follows February ones
unsorted = sales.groupby("product")["amount"].cumsum()
with_totals = sales.assign(
  region_rank=ranks, product_total=total, unsorted_total=unsorted
)
with_totals.sort_values(["product", "month"])

## Part 2 — Aggregations

DVF has one row per transaction (sale, exchange, auction…): its date, its kind, its price (*valeur foncière*), the number of houses, apartments and outbuildings sold, their surfaces, their address… Loaded whole, it takes 4.4 GB in memory.

**Setup**: the cell below only reads what we need, the six useful columns (`columns`) of the sales only (`filters`, applied while reading, even on a column it doesn't load), and replaces the index, the transactions' IDs as strings, with integers (`reset_index`): with pandas 2.2, Colab's, 9,859,162 sales in 169 MB instead of 735 MB. The "Speeding Up Computations" chapter comes back to these memory savings, and its lab measures them.

In [ ]:
# Setup: the six useful columns of the sales only, an integer index
dvf = pd.read_parquet(
  "dvf/dvf-linearized-2014-2022.parquet",
  columns=[
    "annee_mutation",
    "code_departement",
    "valeur_fonciere",
    "nombre_maisons",
    "nombre_appartements",
    "surface_reelle_bati_total",
  ],
  filters=[("nature_mutation", "==", "Vente")],
).reset_index(drop=True)
dvf.info(memory_usage="deep")

### 2.1 Two-Character Département Codes

DVF's département codes have no leading zero (`"1"` for Ain), while the départements table of part 3 has one (`"01"`). `code_departement` is categorical: 97 distinct values, and a small integer per row. Rename its categories, with `cat.rename_categories`, rather than its 9.9 million values, then sort them with `cat.reorder_categories`: the order of the categories is the order of `groupby`'s results.

To check: the categories start with `01`, `02`, `03` and end with `971` to `974`; the column still weighs about 9.4 MB.

**Useful documentation**

- [`Series.cat.rename_categories`](https://pandas.pydata.org/docs/reference/api/pandas.Series.cat.rename_categories.html)
- [`Series.cat.reorder_categories`](https://pandas.pydata.org/docs/reference/api/pandas.Series.cat.reorder_categories.html)
- [`str.zfill`](https://docs.python.org/3/library/stdtypes.html#str.zfill)

In [ ]:
# Your code here

#### Additional hints

The calls and parameters to use:

- `rename_categories` takes a dictionary or a function applied to each category: `lambda c: c.zfill(2)` pads with zeros on the left up to two characters (`"2A"` and `"971"` don't change);
- after the renaming, the categories keep their previous order (`01`, `10`, `11`…, `02`…); `reorder_categories(sorted(...))` takes every category, in the new order;
- with pandas 2.2, Colab's, `replace` on a categorical column is deprecated (`FutureWarning`), and `astype(str).str.zfill(2)` would create 9.9 million strings, about 480 MB.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
codes = dvf["code_departement"].cat.rename_categories(lambda c: ...)
dvf["code_departement"] = codes.cat.reorder_categories(sorted(...))
print(dvf["code_departement"].cat.categories)
print(dvf["code_departement"].memory_usage(deep=True) / 2**20)
```

#### Solution

In [ ]:
codes = dvf["code_departement"].cat.rename_categories(lambda c: c.zfill(2))
dvf["code_departement"] = codes.cat.reorder_categories(sorted(codes.cat.categories))
print(dvf["code_departement"].cat.categories)
print(dvf["code_departement"].memory_usage(deep=True) / 2**20)

### 2.2 The Price per m² of Single-Home Sales

A transaction can sell a whole building: the mean price of a département's transactions says nothing about the price of a home. Keep in `homes` the sales of a single home (a house or an apartment, with or without outbuildings), with the `annee_mutation` and `code_departement` columns, and two new ones:

- `type`, categorical: `"house"` or `"apartment"`;
- `price_m2`: the price divided by the built surface, missing (`NaN`) when it is outside the range from 100 to 50,000 €/m² (zero surface, nominal price, typo).

Then delete `dvf` (`del dvf`) to free its memory.

To check: 6,074,486 sales, 3,226,956 of which are houses; 6,044,907 prices per m², with a median of 2,246 €/m².

**Useful documentation**

- [`Series.between`](https://pandas.pydata.org/docs/reference/api/pandas.Series.between.html)
- [`numpy.where`](https://numpy.org/doc/stable/reference/generated/numpy.where.html)

In [ ]:
# Your code here

#### Additional hints

The calls and parameters to use:

- `dvf["nombre_maisons"] + dvf["nombre_appartements"] == 1`: the mask of the single-home sales;
- `np.where(condition, "house", "apartment")`, inside `pd.Categorical(...)` for a categorical column;
- `price.where(price.between(100, 50_000))`: `NaN` outside the range (bounds included); a zero surface gives an infinite price, outside the range too;
- `dvf.assign(type=..., price_m2=...).loc[mask, columns]`: the new columns, then the rows and columns kept, without changing `dvf`.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
single_home = ...
price = dvf["valeur_fonciere"] / dvf[...]
homes = dvf.assign(
  type=pd.Categorical(np.where(..., "house", "apartment")),
  price_m2=price.where(...),
).loc[single_home, ["annee_mutation", "code_departement", "type", "price_m2"]]
del dvf, price, single_home
print(len(homes), homes["type"].value_counts(), homes["price_m2"].describe())
```

#### Solution

In [ ]:
single_home = dvf["nombre_maisons"] + dvf["nombre_appartements"] == 1
price = dvf["valeur_fonciere"] / dvf["surface_reelle_bati_total"]
homes = dvf.assign(
  type=pd.Categorical(np.where(dvf["nombre_maisons"] == 1, "house", "apartment")),
  price_m2=price.where(price.between(100, 50_000)),
).loc[single_home, ["annee_mutation", "code_departement", "type", "price_m2"]]
del dvf, price, single_home
print(len(homes))
print(homes["type"].value_counts())
print(homes["price_m2"].describe())

### 2.3 Named Aggregations by Département

By département, compute in a single `agg`, with named aggregations:

- `n_sales`, the number of sales, with `size`;
- `n`, the number of usable prices per m², with `count`;
- `median`, their median;
- `iqr`, their interquartile range (third quartile minus first), with a `lambda` function.

Why do `n` and `n_sales` differ? Sort by decreasing median, then plot as horizontal bars the median of the 15 most expensive départements. Why the median rather than the mean?

To check: 97 départements; 6,074,486 sales and 6,044,907 usable prices in all, so 29,579 missing prices, 3,197 of which in Paris; Paris first (median of 9,360 €/m², interquartile range of 3,496 €/m²), ahead of Hauts-de-Seine (6,158 €/m²); Creuse last (684 €/m²).

**Useful documentation**

- [`DataFrame.plot.barh`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.barh.html)

In [ ]:
# Your code here

#### Additional hints

The calls and parameters to use:

- `n_sales=("price_m2", "size")` counts the group's rows, missing prices (`NaN`) included; `n=("price_m2", "count")`, its non-missing values only;
- `iqr=("price_m2", lambda s: s.quantile(0.75) - s.quantile(0.25))`: the function gets a group's `Series`;
- `observed=True`: `code_departement` is categorical;
- `stats[["n_sales", "n"]].sum()`: the totals of both counts;
- `sort_values("median", ascending=False)`, then `head(15)["median"].plot.barh()`; `ax.invert_yaxis()` puts the first one at the top; `plt.show()` shows the figure;
- the median resists the extreme prices that are left (a sale at 50,000 €/m² weighs as much as any other, not 20 times more).

A skeleton to copy into the exercise cell, replacing each `...`:

```python
stats = (
  homes.groupby("code_departement", observed=True)
  .agg(
    n_sales=("price_m2", ...),
    n=("price_m2", ...),
    median=...,
    iqr=("price_m2", lambda s: ...),
  )
  .sort_values(...)
)
print(stats[["n_sales", "n"]].sum())
print(stats.round())
ax = stats.head(15)["median"].plot.barh()
ax.invert_yaxis()
plt.show()
```

#### Solution

In [ ]:
stats = (
  homes.groupby("code_departement", observed=True)
  .agg(
    n_sales=("price_m2", "size"),
    n=("price_m2", "count"),
    median=("price_m2", "median"),
    iqr=("price_m2", lambda s: s.quantile(0.75) - s.quantile(0.25)),
  )
  .sort_values("median", ascending=False)
)
print(stats[["n_sales", "n"]].sum())
print(stats.round())
ax = stats.head(15)["median"].plot.barh(figsize=(6, 5))
ax.invert_yaxis()
ax.set_xlabel("Median price (€/m²)")
ax.set_ylabel("Département")
plt.show()

## Part 3 — Hierarchical Indexes

`regions` maps each département code to its region: a `Series` indexed by code, read from the départements table (as strings, to keep the leading zeros).

In [ ]:
regions = pd.read_csv("departements-france.csv", dtype=str).set_index(
  "code_departement"
)["nom_region"]
regions.head()

### 3.1 The Region of Each Sale

Add to `homes` the categorical `region` column, by translating `code_departement` with `map` and `regions`.

To check: no missing region; 17 regions (Mayotte isn't in DVF, nor are Alsace and Moselle, which have their own land register); 1,172,978 sales in Île-de-France.

In [ ]:
# Your code here

#### Additional hints

The calls and parameters to use:

- `homes["code_departement"].map(regions)`: on a categorical column, `map` only translates the 97 categories; several départements sharing a region, the result is made of strings: `.astype("category")`;
- `isna().sum()`, `nunique()` and `value_counts()` to check;
- this is a join on a single column: `merge` would do the same with a whole table.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
homes["region"] = homes["code_departement"].map(...).astype(...)
print(homes["region"].isna().sum(), homes["region"].nunique())
homes["region"].value_counts()
```

#### Solution

In [ ]:
homes["region"] = homes["code_departement"].map(regions).astype("category")
print(homes["region"].isna().sum(), homes["region"].nunique())
homes["region"].value_counts()

### 3.2 Region, then Département

Group `homes` by region then by département to get `by_dep`, indexed by a two-level `MultiIndex`, with the `n` and `median` columns of 2.3. Then:

1. the départements of Bretagne, with `loc`;
2. the row of Paris (`"75"`), without giving its region, with `xs`;
3. the most expensive département of each region: a sort, then `groupby(level=...)` and `tail`;
4. the number of usable prices per m² of each region, the sum of `n` with `groupby(level=...)`.

To check: 97 rows (also try `observed=False`: how many rows, and why?); Bretagne: 22, 29, 35 and 56; the most expensive: 75 in Île-de-France, 06 in Provence-Alpes-Côte d'Azur, 74 in Auvergne-Rhône-Alpes; 4,589 prices per m² in Guyane.

In [ ]:
# Your code here

#### Additional hints

The calls and parameters to use:

- `homes.groupby(["region", "code_departement"], observed=True).agg(...)`: with `observed=False`, every combination of the categories, 17 × 97 = 1,649 rows, almost all of them empty;
- `by_dep.loc["Bretagne"]`: the first level; `by_dep.xs("75", level="code_departement")`: another level;
- the levels of `by_dep` are categorical: `groupby(level="region", observed=True)`, or else a `FutureWarning` with pandas 2.2, Colab's;
- `by_dep.sort_values("median").groupby(level="region", observed=True).tail(1)`: the last row, the most expensive one, of each region.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
by_dep = homes.groupby([..., ...], observed=True).agg(
  n=("price_m2", "count"), median=("price_m2", "median")
)
print(len(by_dep))
print(by_dep.loc[...])
print(by_dep.xs(..., level=...))
print(by_dep.sort_values(...).groupby(level=..., observed=True).tail(...))
print(by_dep.groupby(level=..., observed=True)["n"].sum())
```

#### Solution

In [ ]:
by_dep = homes.groupby(["region", "code_departement"], observed=True).agg(
  n=("price_m2", "count"), median=("price_m2", "median")
)
print(len(by_dep))
print(by_dep.loc["Bretagne"])
print(by_dep.xs("75", level="code_departement"))
print(by_dep.sort_values("median").groupby(level="region", observed=True).tail(1))
print(by_dep.groupby(level="region", observed=True)["n"].sum())

### 3.3 Houses and Apartments by Region (Optional)

Count the sales by region and by type, move the type to the columns with `unstack`, sort the regions by share of apartments, and plot stacked horizontal bars.

To check: 71% of apartments in Provence-Alpes-Côte d'Azur and Île-de-France, 69% in Corse, 21% in Hauts-de-France.

**Useful documentation**

- [`DataFrame.plot.barh`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.barh.html)

In [ ]:
# Your code here

#### Additional hints

The calls and parameters to use:

- `homes.groupby(["region", "type"], observed=True).size().unstack("type")`: one column per type;
- `by_type["apartment"] / by_type.sum(axis=1)`: the share of apartments of each region;
- `by_type.loc[share.sort_values().index]`: the rows in the order of the shares;
- `plot.barh(stacked=True)`: one bar per row, one color per column.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
by_type = homes.groupby([...], observed=True).size().unstack(...)
share = by_type["apartment"] / ...
print(share.sort_values().round(2))
by_type.loc[share.sort_values().index].plot.barh(stacked=...)
plt.show()
```

#### Solution

In [ ]:
by_type = homes.groupby(["region", "type"], observed=True).size().unstack("type")
share = by_type["apartment"] / by_type.sum(axis=1)
print(share.sort_values().round(2))
ax = by_type.loc[share.sort_values().index].plot.barh(stacked=True, figsize=(7, 6))
ax.set_xlabel("Single-home sales, 2014-2022")
plt.show()

## Part 4 — `transform`, `filter` and `apply`

### 4.1 Deviation from the Département's Median, Outlier Sales

Add to `homes`, with `transform`:

- `deviation`, the relative deviation of the price per m² from the median of its département (0.25: 25% more expensive than that median);
- `outlier`, true when the price per m² is more than three times above or below that median.

Then apply the same rule with the national median: what share of the sales of Paris (75) and Creuse (23) does each rule call outliers?

To check: 4.3% of outlier sales with the per-département rule; Paris, 86% with the national rule against 2.7%; Creuse, 54% against 8.8%.

In [ ]:
# Your code here

#### Additional hints

The calls and parameters to use:

- `homes.groupby("code_departement", observed=True)["price_m2"].transform("median")`: the département's median, on each row of `homes`;
- `ratio = homes["price_m2"] / median`, then `ratio - 1` and `(ratio < 1 / 3) | (ratio > 3)`; a comparison with `NaN` is false;
- `homes["price_m2"].median()`: the national median, a number, which the division broadcasts to every row;
- the mean of a boolean column is the share of `True`: `shares.groupby(homes["code_departement"], observed=True).mean()`, with `shares` a `DataFrame` of both rules.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
median = homes.groupby(...)["price_m2"].transform(...)
ratio = homes["price_m2"] / median
homes["deviation"] = ...
homes["outlier"] = ...
national_ratio = homes["price_m2"] / ...
shares = pd.DataFrame(
  {
    "national": ...,
    "department": homes["outlier"],
  }
)
print(homes["outlier"].mean())
shares.groupby(homes["code_departement"], observed=True).mean().loc[["75", "23"]]
```

#### Solution

In [ ]:
median = homes.groupby("code_departement", observed=True)["price_m2"].transform(
  "median"
)
ratio = homes["price_m2"] / median
homes["deviation"] = ratio - 1
homes["outlier"] = (ratio < 1 / 3) | (ratio > 3)
national_ratio = homes["price_m2"] / homes["price_m2"].median()
shares = pd.DataFrame(
  {
    "national": (national_ratio < 1 / 3) | (national_ratio > 3),
    "department": homes["outlier"],
  }
)
print(homes["outlier"].mean())
shares.groupby(homes["code_departement"], observed=True).mean().loc[["75", "23"]]

### 4.2 `filter` versus a Mask

We only study the price of apartments in the départements with at least 2,000 apartment sales. From `apartments`, keep the sales of those départements in two ways: with `filter`, then with a mask computed by `transform("size")`. Check that both results are equal, time them with `%time`, and list the excluded départements.

To check: 7 départements excluded, 9,540 fewer sales; Creuse has the fewest (571).

In [ ]:
apartments = homes[homes["type"] == "apartment"]
g = apartments.groupby("code_departement", observed=True)
# Your code here

#### Additional hints

The calls and parameters to use:

- `%time x = expression`: the time of one statement;
- `g.filter(lambda d: len(d) >= 2000)`: one Python function call per group;
- `g["price_m2"].transform("size") >= 2000`: the département's number of sales on each row, in a single computation; the mask combines with other conditions (`&`);
- `n_sales = g.size()`, then `n_sales[n_sales < 2000]`.

A skeleton to copy into the exercise cell, below `g`, replacing each `...`:

```python
%time kept = g.filter(lambda d: ...)
%time mask = g["price_m2"].transform(...) >= 2000
print(apartments[mask].equals(kept), len(apartments) - len(kept))
n_sales = g.size()
n_sales[...]
```

#### Solution

In [ ]:
%time kept = g.filter(lambda d: len(d) >= 2000)
%time mask = g["price_m2"].transform("size") >= 2000
print(apartments[mask].equals(kept), len(apartments) - len(kept))
n_sales = g.size()
n_sales[n_sales < 2000].sort_values()

### 4.3 The Most Expensive Sales per m² (Optional)

Find the three sales with the highest price per m² of each département, in two ways:

- `top3_apply`: `apply` of a function that returns the three largest rows of its group (`nlargest`), with `include_groups=False`;
- `top3_sort`: a sort of the whole of `homes` by decreasing price, then `groupby(...).head(3)`.

Time them with `%time`, by département, then by département, year and type. Do they find the same prices? Which one scales better when the number of groups grows, and why?

To check: 291 rows by département (97 groups × 3), 5,236 by département, year and type (1,746 groups, a few of them with fewer than 3 sales), with the same prices (ties can change the rows picked); in general, `apply` is the fastest with 97 groups, the sort with 1,746: the time of `apply` grows with the number of groups, the sort's mostly depends on the number of rows; their ratio varies with the machine and the pandas version.

In [ ]:
def top3_apply(keys): ...


def top3_sort(keys): ...


# Your code here

#### Additional hints

The calls and parameters to use:

- `homes.groupby(keys, observed=True).apply(lambda d: d.nlargest(3, "price_m2"), include_groups=False)`: one Python call per group, 97, then 1,746;
- `homes.sort_values("price_m2", ascending=False).groupby(keys, observed=True).head(3)`: a single sort of the 6 million rows, whatever the groups;
- `np.array_equal(np.sort(a["price_m2"]), np.sort(b["price_m2"]))`: same prices, in any order;
- the built-in method `groupby(keys)["price_m2"].nlargest(3)` is the fastest, but only returns the prices column.

A skeleton to copy into the exercise cell, instead of the two functions, replacing each `...`:

```python
def top3_apply(keys):
  g = homes.groupby(keys, observed=True)
  return g.apply(lambda d: ..., include_groups=False)


def top3_sort(keys):
  ordered = homes.sort_values(...)
  return ordered.groupby(keys, observed=True).head(...)


%time a = top3_apply("code_departement")
%time b = top3_sort("code_departement")
print(len(a), len(b), np.array_equal(np.sort(a["price_m2"]), np.sort(b["price_m2"])))
%time a = top3_apply(["code_departement", "annee_mutation", "type"])
%time b = top3_sort(...)
print(...)
```

#### Solution

In [ ]:
def top3_apply(keys):
  g = homes.groupby(keys, observed=True)
  return g.apply(lambda d: d.nlargest(3, "price_m2"), include_groups=False)


def top3_sort(keys):
  ordered = homes.sort_values("price_m2", ascending=False)
  return ordered.groupby(keys, observed=True).head(3)


%time a = top3_apply("code_departement")
%time b = top3_sort("code_departement")
print(len(a), len(b), np.array_equal(np.sort(a["price_m2"]), np.sort(b["price_m2"])))
%time a = top3_apply(["code_departement", "annee_mutation", "type"])
%time b = top3_sort(["code_departement", "annee_mutation", "type"])
print(len(a), len(b), np.array_equal(np.sort(a["price_m2"]), np.sort(b["price_m2"])))

## Part 5 — Pivot Tables

**New session**: to resume the lab here without redoing parts 2 to 4, run the two cells of "Data and Libraries", then the cell below: it rebuilds `homes` as those parts leave it. If `homes` already exists, it does nothing. Part 6 only needs the first two cells.

In [ ]:
# New session: rebuilds homes as parts 2 to 4 leave it
if "homes" not in globals():
  dvf = pd.read_parquet(
    "dvf/dvf-linearized-2014-2022.parquet",
    columns=[
      "annee_mutation",
      "code_departement",
      "valeur_fonciere",
      "nombre_maisons",
      "nombre_appartements",
      "surface_reelle_bati_total",
    ],
    filters=[("nature_mutation", "==", "Vente")],
  ).reset_index(drop=True)
  codes = dvf["code_departement"].cat.rename_categories(lambda c: c.zfill(2))
  dvf["code_departement"] = codes.cat.reorder_categories(sorted(codes.cat.categories))
  single_home = dvf["nombre_maisons"] + dvf["nombre_appartements"] == 1
  price = dvf["valeur_fonciere"] / dvf["surface_reelle_bati_total"]
  homes = dvf.assign(
    type=pd.Categorical(np.where(dvf["nombre_maisons"] == 1, "house", "apartment")),
    price_m2=price.where(price.between(100, 50_000)),
  ).loc[single_home, ["annee_mutation", "code_departement", "type", "price_m2"]]
  del dvf, price, single_home
  regions = pd.read_csv("departements-france.csv", dtype=str).set_index(
    "code_departement"
  )["nom_region"]
  homes["region"] = homes["code_departement"].map(regions).astype("category")
  median = homes.groupby("code_departement", observed=True)["price_m2"].transform(
    "median"
  )
  ratio = homes["price_m2"] / median
  homes["deviation"] = ratio - 1
  homes["outlier"] = (ratio < 1 / 3) | (ratio > 3)
  del median, ratio

### 5.1 The Median Price by Region and Year

With `pivot_table`, compute `price_by_region`: the median price per m² of `homes` by region (as rows) and by year (as columns), with the totals (`margins`), named `"France"`. `region` and `annee_mutation` are categorical: `observed=True`.

How many prices does each median rest on? Also compute `detail`, the same table with `aggfunc=["median", "count"]`, and display its `count` block: which medians rest on few sales?

To check: 18 rows and 10 columns; the France × France cell is the national median of 2.2, 2,246 €/m²: a total is computed on the sales, not on the cells; Île-de-France goes from 4,036 €/m² in 2014 to 5,022 in 2022; in `detail`, 4,589 prices in Guyane (from 355 to 771 a year) and only 156 in Guadeloupe in 2022.

In [ ]:
# Your code here

#### Additional hints

The calls and parameters to use:

- `index="region"`, `columns="annee_mutation"`, `values="price_m2"`, `aggfunc="median"`;
- `margins=True, margins_name="France"`: a row and a column of totals;
- `aggfunc=["median", "count"]`: one column per function and per year, under a first column level named after the function; `detail["count"]` keeps the `count` block;
- `count`, not `size`: a median only rests on the non-missing prices;
- `.round()` for the display.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
price_by_region = homes.pivot_table(
  index=...,
  columns=...,
  values=...,
  aggfunc=...,
  margins=...,
  margins_name=...,
  observed=True,
)
print(price_by_region.round())
detail = homes.pivot_table(
  index="region",
  columns="annee_mutation",
  values="price_m2",
  aggfunc=[...],
  margins=True,
  margins_name="France",
  observed=True,
)
detail[...]
```

#### Solution

In [ ]:
price_by_region = homes.pivot_table(
  index="region",
  columns="annee_mutation",
  values="price_m2",
  aggfunc="median",
  margins=True,
  margins_name="France",
  observed=True,
)
print(price_by_region.round())
detail = homes.pivot_table(
  index="region",
  columns="annee_mutation",
  values="price_m2",
  aggfunc=["median", "count"],
  margins=True,
  margins_name="France",
  observed=True,
)
detail["count"]

### 5.2 The Year-on-Year Change

Compute `change`, the relative change of each region's median price from one year to the next, with `pct_change(axis=1)`, on `price_by_region` without its totals. Show it as a heatmap, annotated in percent and centered on 0. Which region rose most from 2014 to 2021?

To check: the 2014 column is empty (no previous year); in 2021, up everywhere, from 2% (Corse) to 11% (Pays de la Loire); from 2014 to 2021, Pays de la Loire (+28.6%) and Bretagne (+28.2%) first. Careful with 2022: its first half only.

**Useful documentation**

- [`DataFrame.pct_change`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pct_change.html)

In [ ]:
# Your code here

#### Additional hints

The calls and parameters to use:

- `price_by_region.drop(index="France", columns="France")`: without the totals;
- `pct_change(axis=1)`: (year − previous year) / previous year, along the columns;
- `sns.heatmap(change, annot=True, fmt=".0%", cmap="RdBu_r", center=0)`: `fmt` formats the annotations, `center=0` puts white at 0;
- `price_by_region[2021] / price_by_region[2014] - 1`: the change from 2014 to 2021.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
change = price_by_region.drop(...).pct_change(...)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(change, annot=True, fmt=..., cmap="RdBu_r", center=..., ax=ax)
plt.show()
(price_by_region[2021] / price_by_region[2014] - 1).sort_values()
```

#### Solution

In [ ]:
change = price_by_region.drop(index="France", columns="France").pct_change(axis=1)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(change, annot=True, fmt=".0%", cmap="RdBu_r", center=0, ax=ax)
plt.show()
(price_by_region[2021] / price_by_region[2014] - 1).sort_values()

### 5.3 `pivot` Refuses Duplicates (Optional)

`medians` is the median price by region, year and type of home, in long format. Try to make a region × year table of it with `pivot`: why the error? Count the duplicate (region, year) pairs with `duplicated`. Then:

1. do it with `pivot_table` and `aggfunc="mean"`, as a spreadsheet would, and compare with `price_by_region`: why is this result wrong?
2. reshape `medians` without losing anything, with the type in `pivot`'s index.

To check: `ValueError: Index contains duplicate entries`; 153 duplicates; the mean of the medians of houses and apartments is 83 € over the true median in Bretagne in 2014; 34 rows (17 regions × 2 types).

**Useful documentation**

- [`DataFrame.duplicated`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html)

In [ ]:
medians = (
  homes.groupby(["region", "annee_mutation", "type"], observed=True)["price_m2"]
  .median()
  .reset_index()
)
# Your code here

#### Additional hints

The calls and parameters to use:

- `try:` … `except ValueError as error:` to print the error without stopping the run;
- `medians.duplicated(["region", "annee_mutation"]).sum()`: the rows whose pair was already seen;
- `pivot_table(..., aggfunc="mean", observed=True)`, then the difference with `price_by_region.drop(index="France", columns="France")`;
- the median of all the sales isn't the mean of the medians of two groups of different sizes;
- `pivot(index=["region", "type"], columns="annee_mutation", values="price_m2")`.

A skeleton to copy into the exercise cell, below `medians`, replacing each `...`:

```python
try:
  medians.pivot(index=..., columns=..., values=...)
except ValueError as error:
  print(error)
print(medians.duplicated(...).sum())
means = medians.pivot_table(..., observed=True)
print((means - price_by_region.drop(index="France", columns="France")).round())
medians.pivot(index=[...], columns=..., values=...)
```

#### Solution

In [ ]:
try:
  medians.pivot(index="region", columns="annee_mutation", values="price_m2")
except ValueError as error:
  print(error)
print(medians.duplicated(["region", "annee_mutation"]).sum())
means = medians.pivot_table(
  index="region",
  columns="annee_mutation",
  values="price_m2",
  aggfunc="mean",
  observed=True,
)
print((means - price_by_region.drop(index="France", columns="France")).round())
medians.pivot(
  index=["region", "type"], columns="annee_mutation", values="price_m2"
).round()

## Part 6 — Crosstabs and `melt`

`credit`: 30,000 clients of a Taiwanese bank, from April to September 2005, one per row. The cell below only reads the columns used here, and renames `limit` to `credit_limit`:

- `credit_limit`: the credit limit, in Taiwan dollars;
- `sex`, `education`: coded as integers, translated below (the codes 0, 5 and 6 of `education` aren't documented);
- `age`, in years;
- `delay_apr` to `delay_sep`: each month's payment status, from April to September: -1 paid duly, 1 to 8 months of delay, 9 for nine months and more; -2 and 0 aren't documented;
- `default`: 1 if the client defaulted the next month, 0 otherwise.

In [ ]:
# The file's name, then the name used here
NAMES = {"limit": "credit_limit"}
credit = pd.read_csv(
  "credit-default.csv",
  usecols=[
    "id",
    "age",
    "sex",
    "education",
    "delay_apr",
    "delay_may",
    "delay_jun",
    "delay_jul",
    "delay_aug",
    "delay_sep",
    "default",
    *NAMES,
  ],
).rename(columns=NAMES)
credit["sex"] = credit["sex"].map({1: "male", 2: "female"})
credit["education"] = (
  credit["education"]
  .map({1: "graduate school", 2: "university", 3: "high school", 4: "other"})
  .fillna("unknown")
)
credit.head()

### 6.1 The Default Rate by Education

With `pd.crosstab`, compute the share of clients in default (`default`) by education level (`education`): rows that add up to 1, and the row of totals. Also print the counts: which rates rest on few clients?

To check: 22.1% of defaults in all; 25.2% for high school, 19.2% for graduate school; "other" and "unknown" (5.7% and 7.5%) only have 123 and 345 clients.

In [ ]:
# Your code here

#### Additional hints

The calls and parameters to use:

- `pd.crosstab(credit["education"], credit["default"], ...)`: `Series`, not column names;
- `normalize="index"`: each row adds up to 1; `margins=True`: the `All` row (and column);
- without `normalize`, the counts.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
rate = pd.crosstab(credit["education"], ..., normalize=..., margins=...)
print(rate.round(3))
pd.crosstab(..., margins=True)
```

#### Solution

In [ ]:
rate = pd.crosstab(
  credit["education"], credit["default"], normalize="index", margins=True
)
print(rate.round(3))
pd.crosstab(credit["education"], credit["default"], margins=True)

### 6.2 The Default Rate by Age and Sex

Cut `age` into ten-year bands, from `(20, 30]` to `(50, 60]`, then `(60, 80]`, with `pd.cut`. Cross them with sex: the default rate by band and by sex, with the totals, with `crosstab` and its `values` and `aggfunc` parameters.

To check: ages range from 21 to 79; the lowest rate among the 30-40 year-olds (20.4%), and 18.5% among the women of that age; men default more than women in every band, except the 60-80 year-olds.

In [ ]:
# Your code here

#### Additional hints

The calls and parameters to use:

- `pd.cut(credit["age"], bins=[20, 30, 40, 50, 60, 80])`: right-closed intervals, in an ordered categorical `Series`;
- `pd.crosstab(age_band, credit["sex"], values=credit["default"], aggfunc="mean", margins=True)`: the mean of a column of 0s and 1s is the share of 1s;
- `credit["age"].agg(["min", "max"])` for the bounds.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
print(credit["age"].agg(["min", "max"]))
age_band = pd.cut(credit["age"], bins=...)
pd.crosstab(age_band, ..., values=..., aggfunc=..., margins=True).round(3)
```

#### Solution

In [ ]:
print(credit["age"].agg(["min", "max"]))
age_band = pd.cut(credit["age"], bins=[20, 30, 40, 50, 60, 80])
pd.crosstab(
  age_band, credit["sex"], values=credit["default"], aggfunc="mean", margins=True
).round(3)

### 6.3 Payment Delays, from Wide to Long Format

The six columns `delay_apr` to `delay_sep` are a wide format, one month per column. Turn them into long format with `melt` in `long`: the `id` and `default` columns kept, and two new ones, `month` and `delay`. Remove the `delay_` prefix from the months and make them categorical, in calendar order, `MONTHS`. Group the delays of 4 months and more together, then compute with `pivot_table` the default rate by delay (as rows) and by month (as columns).

To check: 180,000 rows (30,000 × 6); from 2 months of delay on, more than one client in two defaults, against 13 to 20% for the statuses -2 to 0; a one-month delay almost only appears in September, a quirk of this data.

**Useful documentation**

- [`Series.str.removeprefix`](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.removeprefix.html)
- [`Series.clip`](https://pandas.pydata.org/docs/reference/api/pandas.Series.clip.html)

In [ ]:
MONTHS = ["apr", "may", "jun", "jul", "aug", "sep"]
# Your code here

#### Additional hints

The calls and parameters to use:

- `credit.melt(id_vars=["id", "default"], value_vars=[f"delay_{m}" for m in MONTHS], var_name="month", value_name="delay")`;
- `long["month"].str.removeprefix("delay_")`, inside `pd.Categorical(..., categories=MONTHS, ordered=True)`;
- `long["delay"].clip(upper=4)`: 4 and more become 4;
- `pivot_table(index="delay", columns="month", values="default", aggfunc="mean", observed=True)`: `month` is categorical; a cell with no client is `NaN`.

A skeleton to copy into the exercise cell, below `MONTHS`, replacing each `...`:

```python
long = credit.melt(
  id_vars=[...],
  value_vars=[f"delay_{m}" for m in MONTHS],
  var_name=...,
  value_name=...,
)
long["month"] = pd.Categorical(..., categories=MONTHS, ordered=True)
long["delay"] = long["delay"].clip(...)
print(long.shape)
long.pivot_table(index=..., columns=..., values=..., aggfunc=..., observed=True)
```

#### Solution

In [ ]:
long = credit.melt(
  id_vars=["id", "default"],
  value_vars=[f"delay_{m}" for m in MONTHS],
  var_name="month",
  value_name="delay",
)
long["month"] = pd.Categorical(
  long["month"].str.removeprefix("delay_"), categories=MONTHS, ordered=True
)
long["delay"] = long["delay"].clip(upper=4)
print(long.shape)
long.pivot_table(
  index="delay", columns="month", values="default", aggfunc="mean", observed=True
).round(2)

### 6.4 The Default Rate by Delays and Credit Limit (Optional)

`crosstab` crosses `Series` that aren't columns of `credit`: compute two of them, then cross them.

- `n_delays`: the number of months, out of six, in which the client is late (a strictly positive `delay_…` status);
- `quartile`: the quartile of their credit limit, four bands of as many clients, with `pd.qcut`, named `Q1` to `Q4`.

Compute the default rate by number of months of delay (as rows) and by quartile (as columns), with the totals. For the same number of delays, does the credit limit change the risk? Also print the counts: which cells rest on few clients? Why would `pivot_table` be less direct here?

To check: quartiles bounded at 50,000, 140,000 and 240,000 dollars; 19,931 clients never late, 1,341 late all six months; a default rate of 11.7% with no delay, 70.3% with six months; with no delay, 16.7% in the first quartile against 9.3% in the last; the last quartile only has 20 clients with five months of delay.

In [ ]:
# Your code here

#### Additional hints

The calls and parameters to use:

- `credit[[f"delay_{m}" for m in MONTHS]] > 0`: a `DataFrame` of booleans; `.sum(axis=1)` counts each row's `True`; `.rename("n_delays")` names the `Series`, and its name becomes the name of the table's rows;
- `pd.qcut(credit["credit_limit"], 4, labels=["Q1", "Q2", "Q3", "Q4"])`: four bands of roughly equal counts (round limits repeat); without `labels`, intervals, like those of `pd.cut`;
- `credit["credit_limit"].quantile([0.25, 0.5, 0.75])`: the bounds of the quartiles;
- `pd.crosstab(n_delays, quartile, values=credit["default"], aggfunc="mean", margins=True)`: the mean of a column of 0s and 1s is the share of 1s;
- `pivot_table` only takes column names: `n_delays` and `quartile` would first have to be added to `credit`.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
delays = credit[[f"delay_{m}" for m in MONTHS]]
n_delays = (delays > ...).sum(axis=...).rename("n_delays")
quartile = pd.qcut(credit["credit_limit"], ..., labels=["Q1", "Q2", "Q3", "Q4"])
print(credit["credit_limit"].quantile([0.25, 0.5, 0.75]))
delay_rate = pd.crosstab(n_delays, quartile, values=..., aggfunc=..., margins=True)
print(delay_rate.round(3))
pd.crosstab(n_delays, quartile, margins=True)
```

#### Solution

In [ ]:
delays = credit[[f"delay_{m}" for m in MONTHS]]
n_delays = (delays > 0).sum(axis=1).rename("n_delays")
quartile = pd.qcut(credit["credit_limit"], 4, labels=["Q1", "Q2", "Q3", "Q4"])
print(credit["credit_limit"].quantile([0.25, 0.5, 0.75]))
delay_rate = pd.crosstab(
  n_delays, quartile, values=credit["default"], aggfunc="mean", margins=True
)
print(delay_rate.round(3))
pd.crosstab(n_delays, quartile, margins=True)